## Participant Info

In [1]:
import pandas as pd

participant_info_df = pd.read_csv("../participant_info.csv")

In [2]:
participant_info_df.head(5)

,group,fluency,code,sex,age,experiment_date,hometown,region
0,bilingual,10,P1,M,26,20220922,Chiayi,South
1,bilingual,6,P2,F,22,20220922,Tainan,South
2,monolingual,0,P3,F,23,20220923,New Taipei,North
3,monolingual,0,P4,F,22,20220923,New Taipei,North
4,monolingual,0,P5,F,24,20220923,Taichung,Central


In [3]:
import tgt
import pytiers
import parselmouth as plm
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import warnings
import tqdm
import math

def hz_to_semitone(hz, ref): return math.log10(hz / ref) * 39.86

warnings.simplefilter("ignore")

## PsychoPy data processing

### Slopes based on logistic regression

In [4]:
def get_maxslope(x, y, absolute=True, return_xy=False):
    slopes = []
    xs = []
    ys = []
    real_slopes = []

    for x1, y1, x2, y2 in zip(x[:-1], y[:-1], x[1:], y[1:]):
        slope = (y2 - y1) / (x2 - x1)
        real_slopes.append(slope)
        slope = abs(slope)
        xs.append((x2 + x1) / 2)
        ys.append((y2 + y1) / 2)
        slopes.append(slope)
    if absolute:
        max_slope = max(slopes)
    else:
        if len([i for i in real_slopes if i >= 0]) == 0:
            max_slope = -max(slopes)
        else:
            max_slope = max(slopes)
    if return_xy:
        return max_slope, (xs[slopes.index(max(slopes))], ys[slopes.index(max(slopes))])
    return max_slope

In [5]:
data = {
    "tone": [],
    "lang": [],
    "level": [],
    "word": [],
    "word_freq": [],
    "acceptance": [],
    "participant": [],
    "bilingual": [],
    "strict": [],
    "advanced": []
}

In [6]:
word_frequencies = {
    "桌腳": 2/11245330, # Mandarin word frequencies from AS Balanced Corpus: https://asbc.iis.sinica.edu.tw
    "喝酒": 4/11245330,
    "吸管": 19/11245330,
    "發展": 6956/11245330,
    "今晚": 190/11245330,
    "壓力": 1861/11245330,
    "飛彈": 511/11245330,
    "登記": 491/11245330,
    "功課": 398/11245330,
    "衣物": 120/11245330,
    "狡怪": 20/711486, # Southern Min word frequencies from the TAT_MOE Corpus: https://tggl.naer.edu.tw/corpora_applications/new
    "買票": 14/711486,
    "早暗": 9/711486,
    "短褲": 5/711486,
    "你看": 125/711486,
    "水果": 6/711486,
    "米酒": 6/711486,
    "飽滿": 1e-10, # 0 frequencty words are set to 1e-10 to avoid log(0) errors
    "滾水": 42/711486,
    "補繳": 1e-10
}

In [7]:
for lang in ("man", "min"):
    if lang=="man":
        participants = participant_info_df[participant_info_df.group=="monolingual"].code.to_list() + \
                    participant_info_df[participant_info_df.group=="bilingual"].code.to_list()
    if lang=="min":
        participants = participant_info_df[participant_info_df.group=="bilingual"].code.to_list()

    pbar = tqdm.tqdm(participants)
    for participant in pbar: # iterate through all participants

        if lang == "man":
            stimuli = {
                "tone21": ["桌腳", "喝酒", "吸管", "發展", "今晚"],
                "tone51": ["壓力", "飛彈", "登記", "功課", "衣物"],
            }
        elif lang == "min":
            stimuli = {
                "tone21": ["狡怪", "買票", "早暗", "短褲", "你看"],
                "tone51": ["水果", "米酒", "飽滿", "滾水", "補繳"],
            }


        df = pd.read_csv(f'raw_data/{participant}/{["Mandarin", "Min"][lang=="min"]}/{participant}_{["Mandarin", "Min"][lang=="min"]}.csv')

        acceptances = [
            clicked_name == "yes_text_trial_test2"
            for clicked_name in df["mouse_test2.clicked_name"]
            if type(clicked_name) == str
        ]
        words = [
            file.split("/")[-1].split("_")[0]
            for file in list(
                df[
                    [
                        type(clicked_name) == str
                        for clicked_name in df["mouse_test2.clicked_name"]
                    ]
                ]["WAV_file"]
            )
        ]

        levels = [
            int(file.split("level")[-1].replace(".wav", ""))
            for file in list(
                df[
                    [
                        type(clicked_name) == str
                        for clicked_name in df["mouse_test2.clicked_name"]
                    ]
                ]["WAV_file"]
            )
        ]
        
        tones = [
            "tone21" if file.split("WAV/")[-1][:2] in stimuli["tone21"] else "tone51"
            for file in list(
                df[
                    [
                        type(clicked_name) == str
                        for clicked_name in df["mouse_test2.clicked_name"]
                    ]
                ]["WAV_file"]
            )
        ]

        words = [
            file.split("WAV/")[-1][:2]
            for file in list(
                df[
                    [
                        type(clicked_name) == str
                        for clicked_name in df["mouse_test2.clicked_name"]
                    ]
                ]["WAV_file"]
            )
        ]

        word_freqs = [word_frequencies[word] for word in words]

        df_result = pd.DataFrame({"acceptance": acceptances, "tone": tones, "level": levels})

        data["tone"]+=tones
        data["level"]+=levels
        data["word"]+=words
        data["word_freq"]+=word_freqs
        data["acceptance"]+=acceptances
        data["participant"]+=[participant]*len(acceptances)
        data["lang"]+=[lang]*len(acceptances)
        data["bilingual"]+=[participant_info_df.group[list(participant_info_df.code).index(participant)]=="bilingual"]*len(acceptances)
        strict = (participant_info_df.group[list(participant_info_df.code).index(participant)] == "bilingual" and lang == "min") or (participant_info_df.group[list(participant_info_df.code).index(participant)]== "monolingual"and lang == "man")
        data["strict"]+=[strict]*len(acceptances)
        data["advanced"]+=[participant_info_df.fluency[list(participant_info_df.code).index(participant)]>=8]*len(acceptances)

df = pd.DataFrame(data)

100%|██████████| 28/28 [00:00<00:00, 70.52it/s]


In [8]:
df.to_csv("processed_data/acceptance_data_for_logistic_and_probit.csv", index=False)

In [9]:
logistic_regression_df = pd.read_csv("processed_data/all_participant_slopes_glm.csv")

In [11]:
acceptance_slope_data = {
    "tone": [],
    "lang": [],
    "acceptance_slope": [],
    "participant": [],
    "bilingual": [],
    "strict": [],
    "advanced": [],
    "fluency": [],
}

for lang in ("man", "min"):
    if lang=="man":
        participants = participant_info_df[participant_info_df.group=="monolingual"].code.to_list() + \
                    participant_info_df[participant_info_df.group=="bilingual"].code.to_list()
    if lang=="min":
        participants = participant_info_df[participant_info_df.group=="bilingual"].code.to_list()

    pbar = tqdm.tqdm(participants)
    for participant in pbar: # iterate through all participants

        for tone in ("tone51", "tone21"):
            slope = logistic_regression_df[(logistic_regression_df.participant==participant) & (logistic_regression_df.tone==tone) & (logistic_regression_df.lang==lang)]["level.trend"].values[0]

            acceptance_slope_data["tone"] += [tone]
            acceptance_slope_data["lang"] += [lang]
            acceptance_slope_data["acceptance_slope"] += [slope if tone == "tone51" else -slope]
            acceptance_slope_data["participant"] += [participant]
            acceptance_slope_data["bilingual"] += [
                participant_info_df.group[list(participant_info_df.code).index(participant)] == "bilingual"
            ]
            strict = (participant_info_df.group[list(participant_info_df.code).index(participant)] == "bilingual" and lang == "min") or (participant_info_df.group[list(participant_info_df.code).index(participant)]== "monolingual"and lang == "man")
            acceptance_slope_data["strict"] += [strict]
            acceptance_slope_data["advanced"] += [
                participant_info_df.fluency[list(participant_info_df.code).index(participant)] >= 8
            ]
            acceptance_slope_data["fluency"] += [
                participant_info_df.fluency[list(participant_info_df.code).index(participant)]
            ]

df_acceptance_slope = pd.DataFrame(acceptance_slope_data)
df_acceptance_slope.to_csv("concatenated_data/experiment3_acceptance_slope.csv", index=False)

100%|██████████| 28/28 [00:00<00:00, 1095.42it/s]


### Acceptance thresholds based on probit analysis

In [12]:
logistic_regression_df = pd.read_csv("processed_data/all_participant_thresholds_probit.csv")

In [13]:
logistic_regression_df

,participant,lang,tone,threshold_50,slope,intercept
0,P1,min,tone21,8.088661,-1.014677,8.207376
1,P1,min,tone51,5.461646,1.692354,-9.243039
2,P2,min,tone21,6.291227,-1.141624,7.182215
3,P2,min,tone51,5.930986,1.637232,-9.710398
4,P7,min,tone21,7.728021,-0.962178,7.435733
...,...,...,...,...,...,...
137,P41,man,tone51,7.200050,1.062334,-7.648861
138,P42,man,tone21,6.238396,-0.800024,4.990869
139,P42,man,tone51,5.547721,1.609690,-8.930111
140,P43,man,tone21,7.049462,-0.923513,6.510271


In [18]:
acceptance_slope_data = {
    "tone": [],
    "lang": [],
    "acceptance_threshold": [],
    "participant": [],
    "bilingual": [],
    "strict": [],
    "advanced": [],
    "fluency": [],
}

for lang in ("man", "min"):
    if lang=="man":
        participants = participant_info_df[participant_info_df.group=="monolingual"].code.to_list() + \
                    participant_info_df[participant_info_df.group=="bilingual"].code.to_list()
    if lang=="min":
        participants = participant_info_df[participant_info_df.group=="bilingual"].code.to_list()

    pbar = tqdm.tqdm(participants)
    for participant in pbar: # iterate through all participants

        for tone in ("tone51", "tone21"):
            slope = logistic_regression_df[(logistic_regression_df.participant==participant) & (logistic_regression_df.tone==tone) & (logistic_regression_df.lang==lang)]["threshold_50"].values[0]

            acceptance_slope_data["tone"] += [tone]
            acceptance_slope_data["lang"] += [lang]
            acceptance_slope_data["acceptance_threshold"] += [slope if tone == "tone51" else -slope]
            acceptance_slope_data["participant"] += [participant]
            acceptance_slope_data["bilingual"] += [
                participant_info_df.group[list(participant_info_df.code).index(participant)] == "bilingual"
            ]
            strict = (participant_info_df.group[list(participant_info_df.code).index(participant)] == "bilingual" and lang == "min") or (participant_info_df.group[list(participant_info_df.code).index(participant)]== "monolingual"and lang == "man")
            acceptance_slope_data["strict"] += [strict]
            acceptance_slope_data["advanced"] += [
                participant_info_df.fluency[list(participant_info_df.code).index(participant)] >= 8
            ]
            acceptance_slope_data["fluency"] += [
                participant_info_df.fluency[list(participant_info_df.code).index(participant)]
            ]

df_acceptance_slope = pd.DataFrame(acceptance_slope_data)
df_acceptance_slope.to_csv("concatenated_data/experiment3_acceptance_threshold.csv", index=False)

100%|██████████| 28/28 [00:00<00:00, 732.22it/s]


In [15]:
df = pd.read_csv("processed_data/acceptance_data_for_logistic_and_probit.csv")

In [16]:
df

,tone,lang,level,word,word_freq,acceptance,participant,bilingual,strict,advanced
0,tone21,man,10,喝酒,3.557032e-07,False,P3,False,True,False
1,tone21,man,9,發展,6.185679e-04,False,P3,False,True,False
2,tone51,man,5,登記,4.366257e-05,False,P3,False,True,False
3,tone21,man,7,桌腳,1.778516e-07,True,P3,False,True,False
4,tone21,man,8,今晚,1.689590e-05,True,P3,False,True,False
...,...,...,...,...,...,...,...,...,...,...
14195,tone21,min,8,短褲,7.027545e-06,True,P43,True,True,False
14196,tone51,min,6,滾水,5.903138e-05,True,P43,True,True,False
14197,tone51,min,6,水果,8.433054e-06,True,P43,True,True,False
14198,tone21,min,8,早暗,1.264958e-05,True,P43,True,True,False


In [17]:
for lang in ("man", "min"):
    for tone in ("tone21", "tone51"):
        temp = df[(df.lang==lang) & (df.tone==tone)]
        print(f"==={lang} {tone}===")
        for word in temp.word.unique():
            percentage = len(temp[temp.word==word]) / len(temp) * 100
            
            print(f"{word}: {percentage:.2f}%")

===man tone21===
喝酒: 20.35%
發展: 19.88%
桌腳: 19.81%
今晚: 19.67%
吸管: 20.28%
===man tone51===
登記: 20.42%
飛彈: 20.44%
功課: 19.88%
壓力: 19.14%
衣物: 20.12%
===min tone21===
買票: 24.32%
短褲: 24.68%
早暗: 24.43%
狡怪: 26.57%
===min tone51===
米酒: 21.29%
水果: 21.21%
滾水: 19.75%
補繳: 19.11%
飽滿: 18.64%
